# Death Count Extraction: Modern LLM Inference

This notebook evaluates modern decoder models (LLMs) for extracting death counts from conflict event descriptions using zero-shot inference (no fine-tuning).

## Models Evaluated

1. **llama-3.1-8b-instruct** - Open-source benchmark (Meta)
2. **mistral-7b-instruct** - Smaller, efficient model (Mistral AI)
3. **mixtral-8x7b-instruct** - Mixture-of-Experts model (Mistral AI)
4. **flan-t5-xl** - Large instruction-following T5 model (Google)
5. **gpt-4o-mini** - Cheap, strong reference (OpenAI API)
6. **gemini-2.5-flash** - Free, fast baseline (Google API)

## Notebook Structure

1. **Setup & Configuration** - Environment setup, API key configuration
2. **Data Management** - Load validation and test data from previous experiments
3. **Model Configuration** - Setup helper functions and paths
4. **Model Inference** - Run inference on each model and compute metrics
5. **Results Comparison** - Merge results and compare all models

## Notes

- Local models (Llama, Mistral, Mixtral, Flan-T5) run on GPU if available
- API models (GPT-4o-mini, Gemini) require API keys in Colab secrets
- Results are saved in the same format as the smaller models notebook for easy comparison


## 1. Setup and Configuration

### 1.1 Environment Setup

#### 1.1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install dependencies from the cloned repo
%pip install -qU pip setuptools wheel
%pip install -r /content/code-satp/models/count-models/requirements.txt

# 4) Make result directories and add to path
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/death-counts-llms").mkdir(parents=True, exist_ok=True)
pathlib.Path("figures").mkdir(parents=True, exist_ok=True)  # For saving plots
sys.path.append("/content/code-satp/models/count-models")

# 5) GPU check
import torch
print("="*80)
print("✅ SETUP COMPLETE")
print("="*80)
print(f"📂 Cloned repo: /content/code-satp")
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: Running on CPU! Training will be very slow.")
    print("   Go to Runtime > Change runtime type > Hardware accelerator > GPU")
print("="*80)

# 6) Set canonical task name for this notebook
TASK_NAME = "death-counts-llms"


#### 1.1.2 Local Setup

In [ ]:
# # For local development, uncomment these lines:
# import sys
# import os

# # Add utils to path
# sys.path.append('./utils')

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# os.makedirs(RESULTS_PATH, exist_ok=True)
# os.makedirs(os.path.join(RESULTS_PATH, "model_checkpoints"), exist_ok=True)
# os.makedirs("./figures", exist_ok=True)
# os.makedirs("./data", exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# # Set task name
# TASK_NAME = "death-counts"

# print("✅ Local setup complete!")


### 1.2 Import Libraries

In [ ]:
# Core libraries
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# Deep learning
import torch

# Custom utilities
from utils import (
    compute_metrics, print_metrics,
    parse_fatalities,
    time_inference_call,
    load_causal, load_t5,
    run_causal_batch, run_t5_batch,
    run_openai_batch, run_gemini_batch,
    llm_already_done
)
from utils.file_io import get_task_results_dir

# Configuration
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    

### 1.3 Hugging Face Authentication (Llama/Mixtral/Flan-T5)

In [ ]:
# Hugging Face token: Colab secrets first, else local env
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('huggingface_token')
    if HF_TOKEN:
        os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN
        print("✅ Hugging Face token loaded from Colab secrets")
    else:
        print("⚠️  Hugging Face token not found in Colab secrets")
except ImportError:
    HF_TOKEN = os.environ.get('HUGGINGFACE_TOKEN')
    if HF_TOKEN:
        print("✅ Hugging Face token loaded from environment")
    else:
        print("⚠️  Hugging Face token not found in environment")

### 1.4 Open AI and Gemini API Keys

In [ ]:
# Fetch API keys from Colab secrets or environment variables
try:
    # Colab environment – load from secrets panel (🔑 icon)
    from google.colab import userdata

    OPENAI_API_KEY = userdata.get('openai_api_key')
    GEMINI_API_KEY = userdata.get('gemini_api_key')

    print("✅ OpenAI API key loaded from Colab secrets")
    print("✅ Gemini API key loaded from Colab secrets")

except ImportError:
    # Local/offline execution – load from environment variables
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

    print("✅ OpenAI API key loaded from environment")
    print("✅ Gemini API key loaded from environment")

## 2 Data Management

### 2.1 Load Validation and Test Data

Load validation and test data from earlier experiments with smaller models.

In [ ]:
# Establish candidate locations for validation and test data
candidate_dirs = [
    Path("/content/code-satp/models/count-models/data"),  # Colab clone
    Path.cwd() / "models/count-models/data",              # Local execution from repo root
]

data_dir = next((d for d in candidate_dirs if (d / "test.csv").exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        "Could not locate test.csv in /content/code-satp/models/count-models/data or models/count-models/data."
    )

val_path = data_dir / "val.csv"
test_path = data_dir / "test.csv"

if not val_path.exists():
    raise FileNotFoundError(f"Validation split not found at {val_path}")
if not test_path.exists():
    raise FileNotFoundError(f"Test split not found at {test_path}")

# Load validation and test data
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(f"✅ Data loaded from {data_dir}")

### 2.2 Verify Data

Verify the data is loaded correctly.


In [ ]:
# Print summary of validation dataset
print(f"\nTotal incidents in validation dataset: {len(val_df):,}")
print(f"Columns: {val_df.columns.tolist()}")  
print("\nValidation dataset head:")
print(val_df.head(3))

# Print summary of test dataset
print(f"\nTotal incidents in test dataset: {len(test_df):,}")
print(f"Columns: {test_df.columns.tolist()}")  
print("\nTest dataset head:")
print(test_df.head(3))

# Verify required columns exist
required_cols = ['incident_summary', 'total_fatalities']
for split_name, split_df in [('validation', val_df), ('test', test_df)]:
    missing_cols = [col for col in required_cols if col not in split_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in {split_name} data: {missing_cols}")

# Ensure an ID column exists for both splits
if 'incident_number' not in val_df.columns:
    val_df = val_df.reset_index(drop=True).reset_index()
    val_df['incident_number'] = 'val_' + val_df['index'].astype(str)
    val_df = val_df.drop('index', axis=1)

if 'incident_number' not in test_df.columns:
    test_df = test_df.reset_index(drop=True).reset_index()
    test_df['incident_number'] = 'test_' + test_df['index'].astype(str)
    test_df = test_df.drop('index', axis=1)

ID_COL = 'incident_number'
print(f"\n✅ Using '{ID_COL}' as ID column")

# Toggle which split to use downstream
EVAL_SPLIT = "val"  # Options: "test" for inference, "val" for fine-tuning workflows
if EVAL_SPLIT not in {"test", "val"}:
    raise ValueError("EVAL_SPLIT must be either 'test' or 'val'")

df = test_df.copy() if EVAL_SPLIT == "test" else val_df.copy()
split_label = "test" if EVAL_SPLIT == "test" else "validation"
print(f"\n✅ Using {split_label} split for downstream analysis: {len(df):,} incidents")

# Show some examples
print("\n" + "="*80)
print(f"Example incidents from {split_label} split:")
print("="*80)
for i, row in df.head(3).iterrows():
    summary = row['incident_summary'][:150] + "..." if len(row['incident_summary']) > 150 else row['incident_summary']
    print(f"\nID: {row[ID_COL]}")
    print(f"Deaths: {row['total_fatalities']}")
    print(f"Text: {summary}")
    print("-" * 80)

## 3. Model Configuration and Setup

In [ ]:
# Configure paths and directories
results_dir = get_task_results_dir(TASK_NAME, create=True)
OUTPUT_DIR = results_dir

print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"📋 ID column: {ID_COL}")
print(f"📊 Dataset size: {len(df):,} incidents")

# Helper function to run model and save results with evaluation
def run_and_save(model_name: str, outputs: list, df_input: pd.DataFrame, id_col: str, timing: dict = None):
    """
    Parse model outputs, compute metrics, and save results.
    
    Args:
        model_name: Model identifier
        outputs: List of raw model outputs
        df_input: Input dataframe with true labels
        id_col: Name of ID column
        timing: Optional timing dictionary from time_inference_call
    """
    # Parse predictions (parse_fatalities always returns int >= 0)
    parsed = [parse_fatalities(s) for s in outputs]
    
    # Get true labels
    true_labels = df_input['total_fatalities'].values
    
    # Compute extraction success rate (check if output was non-empty)
    extraction_success = [bool(s and s.strip()) for s in outputs]
    
    # Compute metrics
    metrics = compute_metrics(parsed, true_labels, extraction_success)
    
    # Add timing if provided
    if timing:
        metrics['timing'] = timing
        print(f"\n⏱️  Timing: {timing['total_time_seconds']:.2f}s total, "
              f"{timing['time_per_item_seconds']:.3f}s/incident, "
              f"{timing['throughput_items_per_second']:.2f} incidents/s")
    
    # Print metrics
    print(f"\n{'='*80}")
    print(f"{model_name.upper()} Results")
    print(f"{'='*80}")
    print_metrics(metrics, model_name)
    
    # Create results dataframe
    results_df = pd.DataFrame({
        id_col: df_input[id_col].values,
        'incident_summary': df_input['incident_summary'].values,
        'true_label': true_labels,
        f'{model_name}_raw': outputs,
        f'{model_name}_prediction': parsed
    })
    
    # Save results
    results_path = OUTPUT_DIR / f"{model_name}.csv"
    results_df.to_csv(results_path, index=False)
    print(f"\n✅ Saved results to: {results_path}")
    
    # Save metrics
    metrics_path = OUTPUT_DIR / f"{model_name}_metrics.json"
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"✅ Saved metrics to: {metrics_path}")
    
    return results_df, metrics


## 4. Training and Evaluation

### 4.1 Llama-3.1-8B-Instruct

In [ ]:
MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
NAME = "llama3_8b"

if not llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    # Load model
    print("Loading model...")
    tok, mdl = load_causal(MODEL)
    
    # Run inference WITH TIMING
    print("Running inference...")
    texts = df['incident_summary'].tolist()
    outs, timing = time_inference_call(
        run_causal_batch, 
        tok, mdl, texts, 
        max_new_tokens=48
    )
    
    # Save results and compute metrics (with timing)
    results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
    
    # Clean up GPU memory
    del tok, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.2 Mistral-7B-Instruct-v0.3

In [ ]:
MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
NAME = "mistral_7b"

if not llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    # Load model
    print("Loading model...")
    tok, mdl = load_causal(MODEL)
    
    # Run inference WITH TIMING
    print("Running inference...")
    texts = df['incident_summary'].tolist()
    outs, timing = time_inference_call(
        run_causal_batch, 
        tok, mdl, texts, 
        max_new_tokens=48
    )
    
    # Save results and compute metrics (with timing)
    results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
    
    # Clean up GPU memory
    del tok, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.3 Mixtral-8×7B-Instruct-v0.1

In [ ]:
# Note: Mixtral is a large model - may require significant GPU memory
MODEL = "mistralai/Mixtral-8x7B-Instruct-v0.1"
NAME = "mixtral_8x7b"

if not llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    print("⚠️  Warning: This is a large model. May require significant GPU memory.")
    
    # Load model
    print("Loading model...")
    tok, mdl = load_causal(MODEL)
    
    # Run inference WITH TIMING
    print("Running inference...")
    texts = df['incident_summary'].tolist()
    outs, timing = time_inference_call(
        run_causal_batch, 
        tok, mdl, texts, 
        max_new_tokens=48
    )
    
    # Save results and compute metrics (with timing)
    results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
    
    # Clean up GPU memory
    del tok, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.4 Flan-T5-XL

In [ ]:
MODEL = "google/flan-t5-xl"
NAME = "flan_t5_xl"

if not llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    # Load model
    print("Loading model...")
    tok, mdl = load_t5(MODEL)
    
    # Run inference WITH TIMING
    print("Running inference...")
    # Toggle few-shot prompting for T5 from the notebook
    from utils import set_t5_fewshot
    T5_FEWSHOT = False  # set True to enable few-shot prompt
    set_t5_fewshot(T5_FEWSHOT)
    texts = df['incident_summary'].tolist()
    outs, timing = time_inference_call(
        run_t5_batch, 
        tok, mdl, texts, 
        max_new_tokens=16, 
        max_input_tokens=512
    )
    
    # Save results and compute metrics (with timing)
    results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
    
    # Clean up GPU memory
    del tok, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.5 GPT-4o-mini

In [ ]:
NAME = "gpt4o_mini"

if not llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    if not OPENAI_API_KEY:
        print("⚠️  ERROR: OpenAI API key not found!")
        print("   Please add 'openai_api_key' to Colab secrets or set OPENAI_API_KEY environment variable")
    else:
        # Run inference WITH TIMING
        print("Running inference via OpenAI API...")
        texts = df['incident_summary'].tolist()
        outs, timing = time_inference_call(
            run_openai_batch,
            texts,
            api_key=OPENAI_API_KEY,
            model_name="gpt-4o-mini",
            max_tokens=256,
            rate_limit_delay=0.1
        )
        
        # Save results and compute metrics (with timing)
        results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")


### 4.6 Gemini-2.5-Flash

In [ ]:
NAME = "gemini_flash"

if not llm_already_done(NAME, OUTPUT_DIR):
    print(f"\n{'='*80}")
    print(f"Running {NAME}...")
    print(f"{'='*80}")
    
    if not GEMINI_API_KEY:
        print("⚠️  ERROR: Gemini API key not found!")
        print("   Please add 'gemini_api_key' to Colab secrets or set GEMINI_API_KEY environment variable")
    else:
        # Run inference WITH TIMING
        print("Running inference via Gemini API...")
        texts = df['incident_summary'].tolist()
        outs, timing = time_inference_call(
            run_gemini_batch,
            texts,
            api_key=GEMINI_API_KEY,
            model_name="gemini-2.5-flash",
            max_output_tokens=256,
            rate_limit_delay=0.1
        )
        
        # Save results and compute metrics (with timing)
        results_df, metrics = run_and_save(NAME, outs, df, ID_COL, timing=timing)
else:
    print(f"✅ {NAME}: Already completed (loading results...)")
    results_df = pd.read_csv(OUTPUT_DIR / f"{NAME}.csv")
    print(f"   Loaded {len(results_df)} results")

## 5. Merge Completed Runs for Comparison

In [ ]:
## 5. Merge Completed Runs for Comparison

# Start with base dataframe (IDs and true labels)
merged = df[[ID_COL, 'incident_summary', 'total_fatalities']].copy()
merged = merged.rename(columns={'total_fatalities': 'true_label'})

# Model names to merge
model_names = ['llama3_8b', 'mistral_7b', 'mixtral_8x7b', 'flan_t5_xl', 'gpt4o_mini', 'gemini_flash']

# Bring in each model file that exists
print(f"\n{'='*80}")
print("Merging results from all models...")
print(f"{'='*80}")

merged_count = 0
for model_name in model_names:
    model_file = OUTPUT_DIR / f"{model_name}.csv"
    if model_file.exists():
        try:
            tmp = pd.read_csv(model_file)
            # Keep prediction column (may be named differently)
            pred_col = None
            for col in tmp.columns:
                if col.endswith('_prediction') or col == f'{model_name}_fatalities':
                    pred_col = col
                    break
            
            if pred_col:
                # Rename to consistent format
                tmp = tmp[[ID_COL, pred_col]].rename(columns={pred_col: f'{model_name}_pred'})
                merged = merged.merge(tmp, on=ID_COL, how='left')
                merged_count += 1
                print(f"✅ Merged {model_name}")
            else:
                print(f"⚠️  {model_name}: Could not find prediction column")
        except Exception as e:
            print(f"❌ {model_name}: Error merging - {e}")
    else:
        print(f"⏭️  {model_name}: Results file not found, skipping")

print(f"\n✅ Merged {merged_count} models")
print(f"   Total rows: {len(merged)}")
print(f"   Columns: {merged.columns.tolist()}")

# Save merged results
merged_path = OUTPUT_DIR / "merged_results.csv"
merged.to_csv(merged_path, index=False)
print(f"\n✅ Saved merged results to: {merged_path}")

# Compute comparison metrics
print(f"\n{'='*80}")
print("Model Comparison Summary")
print(f"{'='*80}")

comparison_data = []
for model_name in model_names:
    pred_col = f'{model_name}_pred'
    if pred_col in merged.columns:
        # Filter out NaN predictions
        mask = merged[pred_col].notna()
        if mask.sum() > 0:
            preds = merged.loc[mask, pred_col].values
            labels = merged.loc[mask, 'true_label'].values
            
            # Compute metrics
            metrics = compute_metrics(preds, labels)
            overall = metrics.get('overall', metrics)
            
            comparison_data.append({
                'Model': model_name,
                'MAE': overall['mae'],
                'RMSE': overall['rmse'],
                'MdAE': overall['mdae'],
                'Exact Match': overall['exact_match'],
                'Within-1': overall['within_1'],
                'Within-2': overall['within_2'],
                'Non-zero MAE': overall['nonzero_mae'],
                'Extraction Rate': overall['extraction_rate'],
                'N': len(preds)
            })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    print("\n" + comparison_df.to_string(index=False))
    
    # Save comparison
    comparison_path = OUTPUT_DIR / "model_comparison.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print(f"\n✅ Saved comparison metrics to: {comparison_path}")
else:
    print("⚠️  No model results found for comparison")

# Display merged results head
print(f"\n{'='*80}")
print("Merged Results Preview (first 10 rows)")
print(f"{'='*80}")
print(merged.head(10).to_string())
